In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_percentage_error

In [2]:
train_df = pd.read_csv('data/train.csv')

In [3]:
train_df['salary_mean_net']

0        35000.0
1        72500.0
2        11310.0
3        33930.0
4        70000.0
          ...   
49046    47850.0
49047    56550.0
49048    85000.0
49049    30000.0
49050    36540.0
Name: salary_mean_net, Length: 49051, dtype: float64

Выделение целевой переменной. Поскольку метрика соревнования — MAPE, мы логарифмируем таргет. Модели будут оптимизироваться на логарифмированной величине, что делает распределение более нормальным и помогает сгладить штрафы за перепрогноз. Исходный таргет удаляется из датафрейма признаков.

In [4]:
y_train_raw = train_df['salary_mean_net'].values
y_train_log = np.log1p(y_train_raw)
train_df = train_df.drop(columns=['salary_mean_net'])

In [5]:
def calculate_mape(y_true_raw, y_pred_log):
    y_pred_raw = np.expm1(y_pred_log)
    return mean_absolute_percentage_error(y_true_raw, y_pred_raw)

In [6]:
test_df = pd.read_csv('data/test_x.csv')

train_employers = set(train_df['employer_id'].dropna())
test_employers = set(test_df['employer_id'].dropna())

intersection_count = len(train_employers.intersection(test_employers))
new_in_test_count = len(test_employers.difference(train_employers))

intersection_count, new_in_test_count

(3379, 4429)

Используется для предотвращения утечки данных (data leak), чтобы модель не запоминала специфику конкретного работодателя, которого не будет в тесте.

In [8]:
from sklearn.model_selection import GroupKFold

n_splits = 5
kf_group = GroupKFold(n_splits=n_splits)

groups = train_df['employer_id'].fillna('unknown_employer').astype('category').cat.codes
train_df = train_df.select_dtypes(include=[np.number])
cv_splits = list(kf_group.split(train_df, y_train_log, groups=groups))

In [9]:
df = pd.concat([train_df, test_df], ignore_index=True)

Генерация базовых текстовых статистик на основе сырого описания вакансии. Вычисляются длина текста в символах и словах, количество запятых и восклицательных знаков, а также доля заглавных букв.

In [10]:
df['desc_char_len'] = df['raw_description'].fillna('').apply(len)
df['desc_word_len'] = df['raw_description'].fillna('').apply(lambda x: len(x.split()))
df['desc_comma_count'] = df['raw_description'].fillna('').apply(lambda x: x.count(','))
df['desc_excl_count'] = df['raw_description'].fillna('').apply(lambda x: x.count('!'))
df['desc_upper_ratio'] = df['raw_description'].fillna('').apply(lambda x: sum(1 for c in x if c.isupper()) / (len(x) + 1))

Создание признаков на основе текста с помощью TF-IDF и уменьшения размерности (SVD). Текст превращается в разреженную матрицу биграмм, которая затем сжимается до 100 плотных компонент для передачи в градиентный бустинг.

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

corpus = df['lemmaized_wo_stopwords_raw_description'].fillna('')

tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=50000)
tfidf_matrix = tfidf.fit_transform(corpus)

svd = TruncatedSVD(n_components=100, random_state=42)
svd_matrix = svd.fit_transform(tfidf_matrix)

svd_cols = [f'svd_desc_{i}' for i in range(100)]
df[svd_cols] = svd_matrix

/var/folders/dw/k7m_br6s4hz846j1dx_hjjz40000gq/T/ipykernel_43726/2411771253.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[svd_cols] = svd_matrix


Обработка категориальных признаков. Опыт работы переводится в порядковые числа (Ordinal Encoding), так как он имеет естественную иерархию. Остальные важные категории кодируются числами (Label Encoding), чтобы модели могли воспринимать их как категориальные фичи.


In [12]:
from sklearn.preprocessing import LabelEncoder

exp_map = {
    'Нет опыта': 0,
    'От 1 года до 3 лет': 1,
    'От 3 до 6 лет': 2,
    'Более 6 лет': 3
}
df['experience_num'] = df['experience_name'].map(exp_map).fillna(-1)

cat_cols = [
    'unified_address_city', 
    'unified_address_region', 
    'specializations_profarea_name', 
    'professional_roles_name'
]

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].fillna('unknown'))

/var/folders/dw/k7m_br6s4hz846j1dx_hjjz40000gq/T/ipykernel_43726/793975092.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['experience_num'] = df['experience_name'].map(exp_map).fillna(-1)


Разделение объединенного датафрейма обратно на обучающую и тестовую выборки с новыми сгенерированными признаками.

In [13]:
train_df = df.iloc[:len(train_df)].copy()
test_df = df.iloc[len(train_df):].copy()

Обработка списков и массивов. Текстовые представления списков очищаются от лишних символов, после чего считается количество элементов (Count Encoding). Для навыков извлекаются 50 самых популярных значений по всему датасету и создаются бинарные признаки (One-Hot Encoding). Обработка идет на объединенном датафрейме df (train + test).

In [15]:
from collections import Counter

list_cols = ['key_skills_name', 'employer_industries', 'languages_name']

for col in list_cols:
    df[col] = df[col].fillna('').astype(str).str.replace(r'\[|\]|\'', '', regex=True)
    df[f'{col}_count'] = df[col].apply(lambda x: len([i for i in x.split(',') if i.strip()]) if isinstance(x, str) and x.strip() else 0)

all_skills = [skill.strip() for sublist in df['key_skills_name'].dropna().str.split(',') for skill in sublist if skill.strip()]
top_50_skills = [x[0] for x in Counter(all_skills).most_common(50)]

for skill in top_50_skills:
    col_name = f"has_skill_{skill.replace(' ', '_').lower()}"
    df[col_name] = df['key_skills_name'].str.contains(skill, regex=False, case=False).astype(int)

/var/folders/dw/k7m_br6s4hz846j1dx_hjjz40000gq/T/ipykernel_43726/4281875177.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_count'] = df[col].apply(lambda x: len([i for i in x.split(',') if i.strip()]) if isinstance(x, str) and x.strip() else 0)
/var/folders/dw/k7m_br6s4hz846j1dx_hjjz40000gq/T/ipykernel_43726/4281875177.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_count'] = df[col].apply(lambda x: len([i for i in x.split(',') if i.strip()]) if isinstance(x, str) and x.strip() else 0)
/var/folders

In [16]:
train_df = df.iloc[:len(y_train_log)].copy()
test_df = df.iloc[len(y_train_log):].copy()

Создание признаков взаимодействий (Interactions) и расчет медианной зарплаты для них (Target Encoding). Чтобы избежать утечки данных (data leak), медианы для обучающей выборки считаются строго Out-of-Fold (OOF) с использованием переданных сплитов cv_splits. Для тестовой выборки применяются глобальные медианы из всего обучающего набора.

In [17]:
train_df['city_exp'] = train_df['unified_address_city'].astype(str) + "_" + train_df['experience_num'].astype(str)
test_df['city_exp'] = test_df['unified_address_city'].astype(str) + "_" + test_df['experience_num'].astype(str)

train_df['role_exp'] = train_df['professional_roles_name'].astype(str) + "_" + train_df['experience_num'].astype(str)
test_df['role_exp'] = test_df['professional_roles_name'].astype(str) + "_" + test_df['experience_num'].astype(str)

train_df['te_city_exp_median'] = np.nan
train_df['te_role_exp_median'] = np.nan

global_median = np.median(y_train_raw)
global_city_exp = pd.Series(y_train_raw).groupby(train_df['city_exp']).median()
global_role_exp = pd.Series(y_train_raw).groupby(train_df['role_exp']).median()

test_df['te_city_exp_median'] = test_df['city_exp'].map(global_city_exp).fillna(global_median)
test_df['te_role_exp_median'] = test_df['role_exp'].map(global_role_exp).fillna(global_median)

for train_idx, val_idx in cv_splits:
    X_tr, X_val = train_df.iloc[train_idx], train_df.iloc[val_idx]
    y_tr = y_train_raw[train_idx]
    
    city_map = pd.Series(y_tr).groupby(X_tr['city_exp']).median()
    role_map = pd.Series(y_tr).groupby(X_tr['role_exp']).median()
    
    train_df.loc[train_df.index[val_idx], 'te_city_exp_median'] = X_val['city_exp'].map(city_map)
    train_df.loc[train_df.index[val_idx], 'te_role_exp_median'] = X_val['role_exp'].map(role_map)

train_df['te_city_exp_median'] = train_df['te_city_exp_median'].fillna(global_median)
train_df['te_role_exp_median'] = train_df['te_role_exp_median'].fillna(global_median)

Удаление текстовых и вспомогательных колонок перед подачей данных в модель, оставляем только числовые признаки. (Применяем логику select_dtypes, которую вы указали ранее, но на финальном этапе генерации фичей)

In [18]:
train_df = train_df.select_dtypes(include=[np.number])
test_df = test_df.select_dtypes(include=[np.number])

Обучение модели LightGBM на числовых признаках (после SVD и Target Encoding). Модель обучается внутри цикла кросс-валидации с ранней остановкой (early stopping). Предсказания для валидационных фолдов сохраняются для оценки OOF (Out-of-Fold), а предсказания для теста усредняются по всем фолдам.

In [22]:
import lightgbm as lgb
import lightgbm as lgb

# Очищаем названия колонок от спецсимволов
train_df.columns = train_df.columns.str.replace(r'[^\w]', '_', regex=True)
test_df.columns = test_df.columns.str.replace(r'[^\w]', '_', regex=True)

oof_preds_lgb = np.zeros(len(train_df))
test_preds_lgb = np.zeros(len(test_df))

for train_idx, val_idx in cv_splits:
    X_tr, y_tr = train_df.iloc[train_idx], y_train_log[train_idx]
    X_val, y_val = train_df.iloc[val_idx], y_train_log[val_idx]
    
    model_lgb = lgb.LGBMRegressor(
        n_estimators=1500,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1
    )
    
    model_lgb.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    oof_preds_lgb[val_idx] = model_lgb.predict(X_val)
    test_preds_lgb += model_lgb.predict(test_df) / len(cv_splits)

mape_lgb = calculate_mape(y_train_raw, oof_preds_lgb)
oof_preds_lgb = np.zeros(len(train_df))
test_preds_lgb = np.zeros(len(test_df))

for train_idx, val_idx in cv_splits:
    X_tr, y_tr = train_df.iloc[train_idx], y_train_log[train_idx]
    X_val, y_val = train_df.iloc[val_idx], y_train_log[val_idx]
    
    model_lgb = lgb.LGBMRegressor(
        n_estimators=1500,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1
    )
    
    model_lgb.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    oof_preds_lgb[val_idx] = model_lgb.predict(X_val)
    test_preds_lgb += model_lgb.predict(test_df) / len(cv_splits)

mape_lgb = calculate_mape(y_train_raw, oof_preds_lgb)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000378 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 518
[LightGBM] [Info] Number of data points in the train set: 39240, number of used features: 4
[LightGBM] [Info] Start training from score 10.703499
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000295 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 518
[LightGBM] [Info] Number of data points in the train set: 39241, number of used features: 4
[LightGBM] [Info] Start training from score 10.690475
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000300 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 516
[LightGBM] [Info] Number of data points in the train set: 39241, number of used features: 4
[LightGBM] [Info] Start trainin

Подготовка сырых текстовых и категориальных признаков для CatBoost. Поскольку ранее датафрейм был ограничен только числовыми колонками (np.number), необходимые признаки загружаются из исходных файлов напрямую. Пропуски заполняются строковым значением, чтобы CatBoost корректно обработал их внутренними алгоритмами.

In [27]:
from catboost import CatBoostRegressor

raw_train = pd.read_csv('data/train.csv')
raw_test = pd.read_csv('data/test_x.csv')

cat_cols_cb = [
    'experience_name', 'schedule_name', 'unified_address_city', 
    'unified_address_region', 'specializations_profarea_name', 'professional_roles_name'
]
text_cols_cb = ['name_clean', 'lemmaized_wo_stopwords_raw_description']

X_cb = raw_train[cat_cols_cb + text_cols_cb].fillna('unknown')
X_test_cb = raw_test[cat_cols_cb + text_cols_cb].fillna('unknown')

In [24]:
!pip install catboost

  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.9/28.9 MB 26.4 MB/s  0:00:01 eta 0:00:01
Using cached graphviz-0.21-py3-none-any.whl (47 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [catboost]1/2 [catboost]


Обучение модели CatBoost с использованием встроенной обработки категорий и текстов. В модель передаются списки cat_features и text_features. Механика Out-of-Fold и усреднения предсказаний для теста аналогична предыдущему шагу.

In [28]:
oof_preds_cb = np.zeros(len(X_cb))
test_preds_cb = np.zeros(len(X_test_cb))

for train_idx, val_idx in cv_splits:
    X_tr, y_tr = X_cb.iloc[train_idx], y_train_log[train_idx]
    X_val, y_val = X_cb.iloc[val_idx], y_train_log[val_idx]
    
    model_cb = CatBoostRegressor(
        iterations=1500,
        learning_rate=0.05,
        cat_features=cat_cols_cb,
        text_features=text_cols_cb,
        random_seed=42,
        verbose=False
    )
    
    model_cb.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        early_stopping_rounds=50
    )
    
    oof_preds_cb[val_idx] = model_cb.predict(X_val)
    test_preds_cb += model_cb.predict(X_test_cb) / len(cv_splits)

mape_cb = calculate_mape(y_train_raw, oof_preds_cb)


Обучение модели XGBoost на числовых признаках. Работает аналогично LightGBM, использует встроенную раннюю остановку. Выступает отличным дополнением для ансамбля за счет отличий во внутреннем устройстве деревьев и сплитов.

In [30]:
import xgboost as xgb

oof_preds_xgb = np.zeros(len(train_df))
test_preds_xgb = np.zeros(len(test_df))

for train_idx, val_idx in cv_splits:
    X_tr, y_tr = train_df.iloc[train_idx], y_train_log[train_idx]
    X_val, y_val = train_df.iloc[val_idx], y_train_log[val_idx]
    
    model_xgb = xgb.XGBRegressor(
        n_estimators=1500,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1,
        early_stopping_rounds=50
    )
    
    model_xgb.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    oof_preds_xgb[val_idx] = model_xgb.predict(X_val)
    test_preds_xgb += model_xgb.predict(test_df) / len(cv_splits)

mape_xgb = calculate_mape(y_train_raw, oof_preds_xgb)

Разделение большой разреженной матрицы TF-IDF (созданной на этапе 3) на обучающую и тестовую части. Эта матрица не сжата через SVD и содержит десятки тысяч колонок, соответствующих отдельным словам и биграммам.

In [31]:
X_train_sparse = tfidf_matrix[:len(y_train_log)]
X_test_sparse = tfidf_matrix[len(y_train_log):]

Обучение линейной модели Ridge Regression (L2-регуляризация) на огромной разреженной матрице TF-IDF. Линейная модель хорошо находит линейные зависимости редких специфичных слов с зарплатой, которые бустинги (особенно после SVD) могут потерять или проигнорировать.

In [32]:
from sklearn.linear_model import Ridge

oof_preds_ridge = np.zeros(X_train_sparse.shape[0])
test_preds_ridge = np.zeros(X_test_sparse.shape[0])

for train_idx, val_idx in cv_splits:
    X_tr, y_tr = X_train_sparse[train_idx], y_train_log[train_idx]
    X_val = X_train_sparse[val_idx]
    
    model_ridge = Ridge(alpha=1.0, random_state=42)
    model_ridge.fit(X_tr, y_tr)
    
    oof_preds_ridge[val_idx] = model_ridge.predict(X_val)
    test_preds_ridge += model_ridge.predict(X_test_sparse) / len(cv_splits)

mape_ridge = calculate_mape(y_train_raw, oof_preds_ridge)

In [33]:
import optuna

def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 1500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 16, 64),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'random_state': 42,
        'n_jobs': -1
    }
    
    oof_preds = np.zeros(len(train_df))
    
    for train_idx, val_idx in cv_splits:
        X_tr, y_tr = train_df.iloc[train_idx], y_train_log[train_idx]
        X_val, y_val = train_df.iloc[val_idx], y_train_log[val_idx]
        
        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
        )
        
        oof_preds[val_idx] = model.predict(X_val)
        
    return calculate_mape(y_train_raw, oof_preds)

study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=15)

best_lgb_params = study_lgb.best_params

[I 2026-05-15 11:28:27,065] A new study created in memory with name: no-name-3a2e1b8b-8eb8-4135-a59c-076c4edd219d


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000305 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 518
[LightGBM] [Info] Number of data points in the train set: 39240, number of used features: 4
[LightGBM] [Info] Start training from score 10.703499
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000306 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 518
[LightGBM] [Info] Number of data points in the train set: 39241, number of used features: 4
[LightGBM] [Info] Start training from score 10.690475
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000320 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 516
[LightGBM] [Info] Number of data points in the t

[I 2026-05-15 11:28:28,772] Trial 0 finished with value: 0.45358020038058994 and parameters: {'n_estimators': 912, 'learning_rate': 0.05404565403926329, 'num_leaves': 38, 'max_depth': 9, 'subsample': 0.9499803659247976, 'colsample_bytree': 0.8316795468307081, 'reg_alpha': 0.21482210044816807, 'reg_lambda': 0.11778592323282855}. Best is trial 0 with value: 0.45358020038058994.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000403 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 518
[LightGBM] [Info] Number of data points in the train set: 39240, number of used features: 4
[LightGBM] [Info] Start training from score 10.703499
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000286 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 518
[LightGBM] [Info] Number of data points in the train set: 39241, number of used features: 4
[LightGBM] [Info] Start training from score 10.690475
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warnin

[I 2026-05-15 11:28:31,812] Trial 1 finished with value: 0.45420465220292233 and parameters: {'n_estimators': 1331, 'learning_rate': 0.028868549707945994, 'num_leaves': 48, 'max_depth': 9, 'subsample': 0.787445936297902, 'colsample_bytree': 0.8158605073681116, 'reg_alpha': 0.187897178706027, 'reg_lambda': 0.5378080477511998}. Best is trial 0 with value: 0.45358020038058994.


...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2026-05-15 11:29:03,960] Trial 13 finished with value: 0.45583899358243724 and parameters: {'n_estimators': 645, 'learning_rate': 0.021688506272086842, 'num_leaves': 55, 'max_depth': 7, 'subsample': 0.852147860821754, 'colsample_bytree': 0.6176883928543202, 'reg_alpha': 1.2882717925550307, 'reg_lambda': 0.0070057787745757254}. Best is trial 2 with value: 0.4533068166144134.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000286 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 518
[LightGBM] [Info] Number of data points in the train set: 39240, number of used features: 4
[LightGBM] [Info] Start training from score 10.703499
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000295 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 518
[LightGBM] [Info] Number of data points in the train set: 39241, number of used features:

[I 2026-05-15 11:29:05,758] Trial 14 finished with value: 0.4557275664544963 and parameters: {'n_estimators': 1229, 'learning_rate': 0.09290789328085731, 'num_leaves': 57, 'max_depth': 8, 'subsample': 0.7131286767997267, 'colsample_bytree': 0.7705656670444015, 'reg_alpha': 0.009466720676958256, 'reg_lambda': 0.02455306358211875}. Best is trial 2 with value: 0.4533068166144134.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


In [35]:
def objective_cb(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 300, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-1, 10.0, log=True),
        'random_seed': 42,
        'verbose': False
    }
    
    oof_preds = np.zeros(len(X_cb))
    
    for train_idx, val_idx in cv_splits:
        X_tr, y_tr = X_cb.iloc[train_idx], y_train_log[train_idx]
        X_val, y_val = X_cb.iloc[val_idx], y_train_log[val_idx]
        
        model = CatBoostRegressor(**params, cat_features=cat_cols_cb, text_features=text_cols_cb)
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=30)
        
        oof_preds[val_idx] = model.predict(X_val)
        
    return calculate_mape(y_train_raw, oof_preds)

study_cb = optuna.create_study(direction='minimize')
study_cb.optimize(objective_cb, n_trials=6)

best_cb_params = study_cb.best_params

[I 2026-05-15 12:06:26,225] A new study created in memory with name: no-name-a1ed7ab3-a378-4285-8356-f83b8aafbf55
[I 2026-05-15 12:11:29,058] Trial 0 finished with value: 0.2735868263184995 and parameters: {'iterations': 488, 'learning_rate': 0.02659366176484084, 'depth': 7, 'l2_leaf_reg': 7.643636717212251}. Best is trial 0 with value: 0.2735868263184995.
[I 2026-05-15 12:14:31,937] Trial 1 finished with value: 0.2607847613674657 and parameters: {'iterations': 521, 'learning_rate': 0.08226090169421621, 'depth': 6, 'l2_leaf_reg': 0.2885292197672036}. Best is trial 1 with value: 0.2607847613674657.
[I 2026-05-15 12:18:26,808] Trial 2 finished with value: 0.26636370083871347 and parameters: {'iterations': 381, 'learning_rate': 0.057114069909523425, 'depth': 7, 'l2_leaf_reg': 6.823013739296904}. Best is trial 1 with value: 0.2607847613674657.
[I 2026-05-15 12:19:31,020] Trial 3 finished with value: 0.2821061398845193 and parameters: {'iterations': 403, 'learning_rate': 0.03238016761285293

In [36]:
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 1500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'random_state': 42,
        'n_jobs': -1,
        'early_stopping_rounds': 30
    }
    
    oof_preds = np.zeros(len(train_df))
    
    for train_idx, val_idx in cv_splits:
        X_tr, y_tr = train_df.iloc[train_idx], y_train_log[train_idx]
        X_val, y_val = train_df.iloc[val_idx], y_train_log[val_idx]
        
        model = xgb.XGBRegressor(**params)
        
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        
        oof_preds[val_idx] = model.predict(X_val)
        
    return calculate_mape(y_train_raw, oof_preds)

study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=10)

best_xgb_params = study_xgb.best_params

[I 2026-05-15 12:30:51,610] A new study created in memory with name: no-name-9539b846-2e4e-4615-84b9-17cb467931f3
[I 2026-05-15 12:30:53,386] Trial 0 finished with value: 0.45763247137974095 and parameters: {'n_estimators': 1061, 'learning_rate': 0.0721473650137582, 'max_depth': 8, 'subsample': 0.8897681937075096, 'colsample_bytree': 0.87663930551069, 'reg_alpha': 0.008066837529931915, 'reg_lambda': 0.001887350440772734}. Best is trial 0 with value: 0.45763247137974095.
[I 2026-05-15 12:30:55,253] Trial 1 finished with value: 0.45502474626336376 and parameters: {'n_estimators': 773, 'learning_rate': 0.07144193164808534, 'max_depth': 9, 'subsample': 0.8074664048627491, 'colsample_bytree': 0.7819405065065913, 'reg_alpha': 4.424732731308646, 'reg_lambda': 0.03286752308260919}. Best is trial 1 with value: 0.45502474626336376.
[I 2026-05-15 12:30:57,058] Trial 2 finished with value: 0.45627141460236764 and parameters: {'n_estimators': 1334, 'learning_rate': 0.06398864014066633, 'max_depth':


Сбор предсказаний первого уровня. Логарифмированные предсказания моделей переводятся обратно в исходный масштаб (рубли) с помощью экспоненты. Создаются матрицы мета-признаков (OOF для трейна и усредненные для теста), на которых будет обучаться модель второго уровня.
code


In [38]:
oof_meta = pd.DataFrame({
    'lgb': np.expm1(oof_preds_lgb),
    'cb': np.expm1(oof_preds_cb),
    'xgb': np.expm1(oof_preds_xgb),
    'ridge': np.expm1(oof_preds_ridge)
})

test_meta = pd.DataFrame({
    'lgb': np.expm1(test_preds_lgb),
    'cb': np.expm1(test_preds_cb),
    'xgb': np.expm1(test_preds_xgb),
    'ridge': np.expm1(test_preds_ridge)
})

In [41]:
from sklearn.linear_model import LinearRegression

meta_model = LinearRegression(positive=True, fit_intercept=False)
meta_model.fit(oof_meta, y_train_raw)

stacking_oof_preds = meta_model.predict(oof_meta)
stacking_test_preds = meta_model.predict(test_meta)

mape_stacking = mean_absolute_percentage_error(y_train_raw, stacking_oof_preds)
mape_stacking

0.2790129416837708

In [39]:
from scipy.optimize import minimize

def blend_objective(weights):
    blend_preds = np.dot(oof_meta.values, weights)
    return mean_absolute_percentage_error(y_train_raw, blend_preds)

initial_weights = [0.25, 0.25, 0.25, 0.25]
bounds = [(0, 1)] * 4
constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}

res = minimize(
    blend_objective, 
    initial_weights, 
    method='SLSQP', 
    bounds=bounds, 
    constraints=constraints
)

optimal_weights = res.x
blend_oof_preds = np.dot(oof_meta.values, optimal_weights)
blend_test_preds = np.dot(test_meta.values, optimal_weights)

mape_blend = mean_absolute_percentage_error(y_train_raw, blend_oof_preds)

In [45]:
mape_blend

0.25639481765774413

In [42]:
final_test_preds = np.round(blend_test_preds / 10) * 10

min_salary = np.min(y_train_raw)
final_test_preds = np.clip(final_test_preds, a_min=min_salary, a_max=None)

submission = pd.DataFrame({
    'ID': raw_test['id'],
    'salary_mean_net': final_test_preds
})

submission.to_csv('submission.csv', index=False)

In [46]:
print(raw_test.columns.tolist())

['name', 'employer_name', 'experience_name', 'schedule_name', 'key_skills_name', 'accept_handicapped', 'accept_kids', 'unified_address_city', 'unified_address_state', 'unified_address_region', 'unified_address_country', 'specializations_profarea_name', 'professional_roles_name', 'languages_name', 'raw_description', 'raw_branded_description', 'lemmaized_wo_stopwords_raw_description', 'lemmaized_wo_stopwords_raw_branded_description', 'if_foreign_language', 'is_branded_description', 'name_clean', 'employment_name', 'employer_id', 'employer_industries', 'id']
